In [1]:
import os
import random
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from tqdm.auto import tqdm

In [2]:
# =====================================
# Random seed to ensure reproducibility
# =====================================
def set_seed(seed=42):
    # 1. Native python seed:
    random.seed(seed)

    # 2. Evironment python seed:
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. Numpy seed:
    np.random.seed(seed)
    
    # 4. PyTorch seed (CPU)
    torch.manual_seed(seed)
    
    # 5. PyTorch seed (GPU / CUDA)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Si usas múltiples GPUs
    
    # 6. CuDNN deterministic for stability in math operations:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42) # Call seed function

In [3]:
# ==============================================================================
# 1. Hardware Configuration and Parameters
# ==============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TARGET_SIZE = 256
BATCH_SIZE = 32
NUM_IMAGES = 1000  # Evaluated sample size per group

# Define paths for all datasets
PATH_REAL_OK = f"../data/processed/casting/casting_{TARGET_SIZE}x{TARGET_SIZE}/ok_front/images/"
PATH_REAL_DEF = f"../data/processed/casting/casting_{TARGET_SIZE}x{TARGET_SIZE}/def_front/images/"

PATH_CVAE_OK = "../data/processed/generated_casting/cvae_ok/"
PATH_CVAE_DEF = "../data/processed/generated_casting/cvae_def/"

PATH_SD_OK = "../data/processed/generated_casting/sd_ok/"
PATH_SD_DEF = "../data/processed/generated_casting/sd_def/"

In [4]:
# ==============================================================================
# 2. Custom Dataset for Scalable Image Loading
# ==============================================================================
class EvaluationDataset(Dataset):
    """
    Loads images from a specific directory, resizing them to the target resolution
    and returning tensors optimized for metric evaluations.
    """
    def __init__(self, directory, size=256, max_samples=1000):
        self.directory = directory
        self.files = [f for f in os.listdir(directory) if f.endswith(('.png', '.jpeg', '.jpg'))][:max_samples]
        
        # Transform for FID (Requires torch.uint8, range [0, 255])
        self.transform_fid = T.Compose([
            T.Resize((size, size)),
            T.PILToTensor()
        ])
        
        # Transform for LPIPS (Requires torch.float32, normalized to [0, 1] or [-1, 1])
        self.transform_lpips = T.Compose([
            T.Resize((size, size)),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # Scales to [-1, 1]
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.directory, self.files[idx])
        image = Image.open(img_path).convert("RGB")
        
        return {
            "fid_tensor": self.transform_fid(image),
            "lpips_tensor": self.transform_lpips(image)
        }

In [5]:
# ==============================================================================
# 3. Metric Core Computation Engine
# ==============================================================================
def compute_1to1_metrics(real_dir, fake_dir, description="Comparison"):
    """
    Computes both FID and LPIPS scores between a real baseline directory 
    and a generated dataset directory using optimized batching.
    """
    print(f"\n--- Running Evaluation: {description} ---")
    
    # Initialize datasets and dataloaders
    dataset_real = EvaluationDataset(real_dir, size=TARGET_SIZE, max_samples=NUM_IMAGES)
    dataset_fake = EvaluationDataset(fake_dir, size=TARGET_SIZE, max_samples=NUM_IMAGES)
    
    loader_real = DataLoader(dataset_real, batch_size=BATCH_SIZE, shuffle=False)
    loader_fake = DataLoader(dataset_fake, batch_size=BATCH_SIZE, shuffle=False)
    
    # Initialize torchmetrics objects
    # feature=2048 targets the final pooling layer of pre-trained Inception-v3
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)
    # net_type='vgg' leverages VGG16 features for structural similarity
    lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type='vgg').to(device)
    
    lpips_values = []
    
    # Step 1: Update metric states with the real distribution
    for batch in tqdm(loader_real, desc="Processing Real Baseline"):
        real_fid = batch["fid_tensor"].to(device)
        fid_metric.update(real_fid, real=True)
        
    # Step 2: Update metric states with the synthetic distribution and compute LPIPS
    # zip ensures a direct 1:1 pairwise assessment across batches
    for batch_real, batch_fake in tqdm(zip(loader_real, loader_fake), total=len(loader_fake), desc="Processing Synthetic Set"):
        fake_fid = batch_fake["fid_tensor"].to(device)
        fid_metric.update(fake_fid, real=False)
        
        # LPIPS computes distance between paired patches
        real_lpips = batch_real["lpips_tensor"].to(device)
        fake_lpips = batch_fake["lpips_tensor"].to(device)
        pair_size = min(real_lpips.shape[0], fake_lpips.shape[0])
        real_lpips = real_lpips[:pair_size]
        fake_lpips = fake_lpips[:pair_size]
        
        with torch.no_grad():
            lpips_dist = lpips_metric(real_lpips, fake_lpips)
            lpips_values.append(lpips_dist.item())
            
    # Step 3: Final execution and retrieval of scores
    final_fid = fid_metric.compute().item()
    final_lpips = np.mean(lpips_values)
    
    print(f"Results for {description}:")
    print(f"  -> FID Score:  {final_fid:.4f}")
    print(f"  -> LPIPS Score: {final_lpips:.4f}")
    
    return final_fid, final_lpips

In [6]:
# ==============================================================================
# 4. Main Evaluation Execution Pipeline
# ==============================================================================
if __name__ == "__main__":
    print(f"Initializing Metric Pipeline on Device: {device.type.upper()}")
    print(f"Target Configuration: {NUM_IMAGES} images per class at {TARGET_SIZE}x{TARGET_SIZE} resolution.")
    
    results = {}
    
    # --------------------------------------------------------------------------
    # AC-CVAE-GAN Evaluations
    # --------------------------------------------------------------------------
    results["cvae_ok"] = compute_1to1_metrics(
        real_dir=PATH_REAL_OK, fake_dir=PATH_CVAE_OK, description="AC-CVAE-GAN [OK Class]"
    )
    results["cvae_def"] = compute_1to1_metrics(
        real_dir=PATH_REAL_DEF, fake_dir=PATH_CVAE_DEF, description="AC-CVAE-GAN [Defect Class]"
    )
    
    # --------------------------------------------------------------------------
    # SD-LoRA-ControlNet Evaluations
    # --------------------------------------------------------------------------
    results["sd_ok"] = compute_1to1_metrics(
        real_dir=PATH_REAL_OK, fake_dir=PATH_SD_OK, description="SD-LoRA-ControlNet [OK Class]"
    )
    results["sd_def"] = compute_1to1_metrics(
        real_dir=PATH_REAL_DEF, fake_dir=PATH_SD_DEF, description="SD-LoRA-ControlNet [Defect Class]"
    )
    
    # --------------------------------------------------------------------------
    # Final Summary Table Output
    # --------------------------------------------------------------------------
    print("\n" + "="*50)
    print("FINAL QUANTITATIVE ASSESSMENT SUMMARY")
    print("="*50)
    print(f"{'Experiment Architecture / Class':<40} | {'FID':<8} | {'LPIPS':<6}")
    print("-"*50)
    for experiment, scores in results.items():
        print(f"{experiment:<40} | {scores[0]:<8.4f} | {scores[1]:<6.4f}")
    print("="*50)

Initializing Metric Pipeline on Device: CUDA
Target Configuration: 1000 images per class at 256x256 resolution.

--- Running Evaluation: AC-CVAE-GAN [OK Class] ---


Processing Real Baseline:   0%|          | 0/17 [00:00<?, ?it/s]

Processing Synthetic Set:   0%|          | 0/32 [00:00<?, ?it/s]

Results for AC-CVAE-GAN [OK Class]:
  -> FID Score:  209.3634
  -> LPIPS Score: 0.4619

--- Running Evaluation: AC-CVAE-GAN [Defect Class] ---


Processing Real Baseline:   0%|          | 0/25 [00:00<?, ?it/s]

...

Processing Synthetic Set:   0%|          | 0/32 [00:00<?, ?it/s]

Results for SD-LoRA-ControlNet [OK Class]:
  -> FID Score:  67.6461
  -> LPIPS Score: 0.2590

--- Running Evaluation: SD-LoRA-ControlNet [Defect Class] ---


Processing Real Baseline:   0%|          | 0/25 [00:00<?, ?it/s]

Processing Synthetic Set:   0%|          | 0/32 [00:00<?, ?it/s]

Results for SD-LoRA-ControlNet [Defect Class]:
  -> FID Score:  96.1679
  -> LPIPS Score: 0.4322

FINAL QUANTITATIVE ASSESSMENT SUMMARY
Experiment Architecture / Class          | FID      | LPIPS 
--------------------------------------------------
cvae_ok                                  | 209.3634 | 0.4619
cvae_def                                 | 180.7291 | 0.4671
sd_ok                                    | 67.6461  | 0.2590
sd_def                                   | 96.1679  | 0.4322
